# Iteration 4: agreement inside the escalation, between queries far apart in heading

**Loss targeted:** the decision on escalated photos (still skipped after the 6 standard queries; about 10 % of photos).

**Hypothesis.** Test_Iteration_3 rejected agreement inside the 27-query escalation: queries 2.5 degrees apart see nearly the same picture and land on
the same false place together (one NUDT wrong answer had 6 agreeing queries). If only queries **at least 5 or 7.5 degrees apart** count, the
pictures differ enough (a 7.5 degree turn moves a cell at the mean radius by more than 2 cells) that a shared chance hit becomes unlikely, while a
right place is still found by queries on both sides of the true heading.

**Test (offline).** `train/checks/replay.decide_spaced`: today's rule, plus an escalated photo below bar 11 is answered when >= n queries with
pairwise heading gaps >= g agree (within 1 sigma, each with >= m inliers). Chosen on official + NUDT 101, reported on NUDT 202 / 303.

In [ ]:
import os, sys
from pathlib import Path
import numpy as np, pandas as pd
ROOT = Path.cwd()
while not (ROOT / "single_photo_pipeline.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT / "src")); os.chdir(ROOT)
pd.set_option("display.width", 220)
from train.checks.replay import load_set, decide_current, decide_spaced, score, paired_bootstrap
DUMPS = ROOT / "tests/Test_Iteration_6/results/rich_dump"
SET_NAMES = {"official": "official", "nudt101": "nudt_seed101", "nudt202": "nudt_seed202", "nudt404": "nudt_fresh404"}
sets = {n: load_set(DUMPS / f) for n, f in SET_NAMES.items() if (DUMPS / f).exists()}
print({k: len(v) for k, v in sets.items()})

In [ ]:
rows = []
base = {n: pd.DataFrame([dict(photo=i, **decide_current(p)) for i, p in ph.items()]) for n, ph in sets.items()}
for n_agree in (2, 3):
    for gap in (5.0, 7.5, 10.0):
        for min_inl in (3, 5):
            for name, ph in sets.items():
                t = pd.DataFrame([dict(photo=i, **decide_spaced(p, escalation_agree=n_agree, escalation_gap=gap, escalation_min_inliers=min_inl))
                                  for i, p in ph.items()])
                g = paired_bootstrap(base[name].points.to_numpy(), t.points.to_numpy())
                rows.append(dict(n_agree=n_agree, gap=gap, min_inl=min_inl, set=name, score=round(score(t.points), 2),
                                 today=round(score(base[name].points), 2), gain="%+.2f [%+.2f, %+.2f]" % g, wrong=int((t.points < 0).sum()),
                                 extra_answers=int((t.stage == "esc_agree").sum()),
                                 extra_right=int(((t.stage == "esc_agree") & (t.points > 0)).sum())))
res = pd.DataFrame(rows)
display(res.pivot_table(index=["n_agree", "gap", "min_inl"], columns="set", values=["score", "wrong", "extra_answers"], aggfunc="first"))
res.to_csv(ROOT / "tests/Test_Iteration_6/results/it4_spaced_agreement.csv", index=False)